<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-06/NB06_systematicity.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 06: Symbolic and Connectionist AI: Systematicity and Compositionality

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-06/lab.html>.

## Overview

Two traditions have competed to explain intelligence: symbolic systems that manipulate structured representations and connectionist networks that learn distributed ones. This week examines their central dispute, whether networks can explain the systematicity of thought, and the empirical tests that now bear on it [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to contrast symbolic and connectionist architectures, to reconstruct the systematicity argument of Fodor and Pylyshyn and Smolensky's reply, to explain why compositional generalisation tests such as SCAN bear on the argument, and to evaluate recent evidence and neurosymbolic proposals.

## Two traditions

Symbolic AI treats intelligence as the manipulation of structured symbols by rules, in the tradition of the physical symbol system hypothesis from Week 3. Connectionism treats it as the result of learning in networks of simple units whose knowledge is stored in the strengths of their connections. The volumes on parallel distributed processing by Rumelhart, McClelland and colleagues made connectionism a serious rival in the 1980s by showing that networks could learn, generalise and degrade gracefully [4]. Present deep learning systems, including language models, descend from this tradition.

## The systematicity argument

Fodor and Pylyshyn argued that human thought has two properties that a cognitive architecture must explain [1]. Thought is productive: People can think indefinitely many new thoughts. Thought is systematic: Anyone who can think that John loves Mary can also think that Mary loves John. Classical architectures explain both properties, because their representations have a combinatorial syntax and semantics: the same parts are recombined by the same rules. Connectionist networks, they argued, face a dilemma. Either they implement a classical architecture, in which case they offer no new theory of cognition, or they do not, in which case they fail to explain systematicity.

Smolensky replied that connectionist models work at a subsymbolic level, below the level of concepts, and that structured representations can be encoded in distributed patterns of activity [2]. On his proposal, symbolic descriptions are approximations of the underlying dynamics rather than the mechanism itself. The dispute therefore concerns explanation as well as performance: whether systematicity is guaranteed by the architecture or merely possible after suitable training.

## Empirical tests

Lake and Baroni turned the argument into an experiment [3]. Their SCAN benchmark maps commands such as jump twice and walk left to action sequences. Sequence-to-sequence networks generalised well when test commands were drawn at random, but failed when a primitive such as jump had been seen only in isolation during training and had to be combined with modifiers at test time. People, by contrast, generalise from a single example. Marcus used such results in a broader critique that listed the limitations of deep learning, including limited transfer and the absence of hierarchical structure [5].

Later work complicated the picture. Lake and Baroni showed that a standard neural network trained with meta-learning for compositionality, across many tasks that each required systematic generalisation, reached human-like performance on new instruction-learning tasks [6]. Systematicity, on this evidence, is achievable by networks but depends on how they are trained. Figure 6.1 reproduces the original contrast on a small command language.

![Figure 6.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-06/figures/w06_fig1.png)

*Figure 6.1. Exact-match accuracy of a symbolic interpreter and a multilayer perceptron on a random split and on a split in which the primitive jump appears only alone during training [3].*

## Hybrid prospects

d'Avila Garcez and Lamb described neurosymbolic AI as a third wave that integrates learning with reasoning over structured representations [7]. For philosophy of mind, the lesson of the debate is that systematicity has become an empirical and architecture-sensitive question. The lab compares three systems on the same command language: a hand-written interpreter, a learner that memorises examples and a learner that acquires word meanings but composes them with fixed grammatical rules. The pattern of their successes and failures shows what compositional structure contributes.

> **Pause and reflect.** If a network becomes systematic only after training on many systematic tasks, has it explained systematicity or inherited it from its training?

# Hands-on lab

The notebook generates all commands of the small language, trains a multilayer perceptron that predicts the action at every output position, and compares its exact-match accuracy with a symbolic interpreter on a random split and on the held-out primitive split [1, 3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. The command language

In [ ]:
import warnings
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
warnings.filterwarnings("ignore")

PRIMS = {"walk": "WALK", "run": "RUN", "jump": "JUMP", "look": "LOOK"}
DIRS = {"": [], "left": ["LTURN"], "right": ["RTURN"]}
MODS = {"": 1, "twice": 2, "thrice": 3}

def interpret(cmd):
    out = []
    for phrase in cmd.split(" and "):
        w = phrase.split()
        d = next((x for x in w if x in DIRS and x), "")
        m = next((x for x in w if x in MODS and x), "")
        out += (DIRS[d] + [PRIMS[w[0]]]) * MODS[m]
    return out

phrases = [" ".join(x for x in (p, d, m) if x) for p in PRIMS for d in DIRS for m in MODS]
COMMANDS = phrases + [a + " and " + b for a in phrases for b in phrases]
print(len(COMMANDS), "commands, for example:", COMMANDS[40], "->", interpret(COMMANDS[40]))

## 2. Encoding and a position-wise network

Each input word is one-hot encoded at its position. The network predicts one action token, or padding, at each of twelve output positions.

In [ ]:
VOCAB = sorted({w for c in COMMANDS for w in c.split()})
TOKS = ["WALK", "RUN", "JUMP", "LOOK", "LTURN", "RTURN", "PAD"]

def encode(cmds, max_len=7):
    X = np.zeros((len(cmds), max_len * len(VOCAB)))
    for r, c in enumerate(cmds):
        for i, w in enumerate(c.split()):
            X[r, i * len(VOCAB) + VOCAB.index(w)] = 1
    return X

def targets(cmds, max_out=12):
    Y = np.full((len(cmds), max_out), TOKS.index("PAD"))
    for r, c in enumerate(cmds):
        for i, t in enumerate(interpret(c)):
            Y[r, i] = TOKS.index(t)
    return Y

def train_and_test(train_cmds, test_cmds):
    net = MultiOutputClassifier(MLPClassifier(hidden_layer_sizes=(64,), max_iter=400, random_state=0))
    net.fit(encode(train_cmds), targets(train_cmds))
    return net.predict(encode(test_cmds))

## 3. Random split versus held-out primitive

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(COMMANDS))
random_train, random_test = [COMMANDS[i] for i in order[:1000]], [COMMANDS[i] for i in order[1000:]]
jump_train = [c for c in COMMANDS if "jump" not in c or c == "jump"]
jump_test = [c for c in COMMANDS if "jump" in c and c != "jump"]
pred_random = train_and_test(random_train, random_test)
pred_jump = train_and_test(jump_train, jump_test)

### Your turn, exercise 1

Write `exact_match(pred, cmds)` that returns the share of commands whose predicted sequence equals the target sequence at every position, and report it for both splits.

In [ ]:
def exact_match(pred, cmds):
    # Your code here
    return None

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _exact_reference(pred, cmds):
    return float(np.mean(np.all(pred == targets(cmds), axis=1)))

In [ ]:
check("Exercise 1", exact_match(pred_random, random_test), _exact_reference(pred_random, random_test))
print("random split:", round(_exact_reference(pred_random, random_test), 3), " held-out primitive:", round(_exact_reference(pred_jump, jump_test), 3))

### Your turn, exercise 2

Design a length split: train on commands whose action sequence has at most six tokens and test on longer ones. Report the exact-match accuracy of the network and explain in two sentences how the result bears on the systematicity argument.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _length_split_reference():
    tr = [c for c in COMMANDS if len(interpret(c)) <= 6]
    te = [c for c in COMMANDS if len(interpret(c)) > 6]
    return _exact_reference(train_and_test(tr, te), te)

In [ ]:
length_accuracy = None  # your computation
check("Exercise 2", length_accuracy, _length_split_reference())

## Visual exploration: Systematic generalisation and exposure

A small network learns to add two symbols encoded one-hot. All combinations that contain the symbol 7 are held out except k of them, and accuracy is measured on the unseen combinations with 7. Change the task or the network size and compare.

In [ ]:
import warnings
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)
S = 10
pairs = [(a, b) for a in range(S) for b in range(S)]

def enc(a, b):
    v = np.zeros(2 * S); v[a] = 1; v[S + b] = 1
    return v

held, rest = [p for p in pairs if 7 in p], [p for p in pairs if 7 not in p]
ks, accs = [0, 1, 2, 4, 8, 12], []
for k in ks:
    idx = set(rng.choice(len(held), k, replace=False).tolist()) if k else set()
    train = rest + [held[i] for i in idx]
    test = [p for j, p in enumerate(held) if j not in idx]
    m = MLPRegressor(hidden_layer_sizes=(32,), max_iter=5000, random_state=SEED)
    m.fit(np.array([enc(*p) for p in train]), np.array([sum(p) for p in train]))
    pred = m.predict(np.array([enc(*p) for p in test]))
    accs.append(np.mean(np.abs(pred - np.array([sum(p) for p in test])) < 0.5))
plt.bar([str(k) for k in ks], accs)
plt.xlabel("training examples that contain the symbol 7"); plt.ylabel("accuracy on unseen combinations with 7")
plt.title("Systematic generalisation depends on exposure"); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-06/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which system in the lab behaves most like a person learning a new verb? What does that suggest about the architecture of human cognition?
2. Does the success of meta-learning answer Fodor and Pylyshyn's dilemma, or does it choose its first horn?
3. Describe a compositional generalisation that matters in your field and how you would test a model for it.

## Weekly task and submission

Write about 500 words evaluating the systematicity argument in the light of the notebook results and the evidence of Lake and Baroni. State which horn of the dilemma present neural networks fall under, if any. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Neurosymbolic systems and the systematicity challenge.** Review how neurosymbolic and meta-learning approaches respond to the systematicity argument and assess which horn of the dilemma they choose [1, 6, 7].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Fodor, J. A., & Pylyshyn, Z. W. (1988). Connectionism and cognitive architecture: A critical analysis. *Cognition*, 28(1-2), 3-71. <https://doi.org/10.1016/0010-0277(88)90031-5>

[2] Smolensky, P. (1988). On the proper treatment of connectionism. *Behavioral and Brain Sciences*, 11(1), 1-23. <https://doi.org/10.1017/S0140525X00052432>

[3] Lake, B., & Baroni, M. (2018). Generalization without systematicity: On the compositional skills of sequence-to-sequence recurrent networks. In *Proceedings of the 35th International Conference on Machine Learning, PMLR 80* (pp. 2873-2882). <https://arxiv.org/abs/1711.00350>

[4] Rumelhart, D. E., McClelland, J. L., & the PDP Research Group (1986). *Parallel Distributed Processing: Explorations in the Microstructure of Cognition, Volume 1: Foundations*. MIT Press.

[5] Marcus, G. (2018). Deep learning: A critical appraisal. arXiv preprint arXiv:1801.00631. <https://arxiv.org/abs/1801.00631>

[6] Lake, B. M., & Baroni, M. (2023). Human-like systematic generalization through a meta-learning neural network. *Nature*, 623(7985), 115-121. <https://doi.org/10.1038/s41586-023-06668-3>

[7] d'Avila Garcez, A., & Lamb, L. C. (2023). Neurosymbolic AI: The 3rd wave. *Artificial Intelligence Review*, 56(11), 12387-12406. <https://doi.org/10.1007/s10462-023-10448-w>